# Computer Vision Lab 04 — Complete Explained Notebook

**Based directly on the uploaded NUces AI-4002 Computer Vision Lab 04 manual.**

This notebook includes the manual's code examples rewritten with detailed, easy comments, plus parameter references and scenario-based guidance.

**Before running:** put your image in the notebook folder and name it `image.jpg`, or change `IMAGE_PATH` below.

In [ ]:
# CELL 1 — INSTALL REQUIRED LIBRARIES
# Run this once in a fresh notebook.
# OpenCV -> image reading, filtering, convolution, Sobel/Scharr/Canny
# NumPy -> arrays and numerical calculations
# Matplotlib -> plotting
# scikit-image -> HOG and LBP

!pip install opencv-python numpy matplotlib scikit-image

In [ ]:
# CELL 2 — IMPORTS AND COMMON SETTINGS
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern
from skimage import exposure

IMAGE_PATH = "image.jpg"

# Read as grayscale. Most feature-extraction examples in this manual
# operate on grayscale images.
image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

if image is None:
    raise FileNotFoundError(
        f"Could not read {IMAGE_PATH}. Put it in the notebook folder "
        "or change IMAGE_PATH."
    )

print("Image shape:", image.shape)
print("Image data type:", image.dtype)

## SCENARIO GUIDE — WHICH TECHNIQUE SHOULD I CHOOSE?

| Situation | Technique | Main reason |
|---|---|---|
| Object/person shape and edge structure | **HOG** | Local gradient directions |
| Texture / surface pattern | **LBP** | Pixel-vs-neighbor comparisons |
| Overall color distribution | **Color Histogram** | Counts channel intensities |
| Dominant edge directions | **HED** | Edge orientation histogram |
| Distribution of intensity gradients | **HIG** | Gradient orientation distribution |
| Texture activity / complexity | **Texture Energy** | Squared local intensities |
| Texture variation | **Local Contrast / Std** | Local intensity variation |
| Reduce noise | **Gaussian Blur** | Smooths image |
| Simple averaging | **Box Blur** | Neighborhood average |
| Horizontal/vertical gradients | **Sobel** | Gradient operator |
| Accurate 3×3 derivative | **Scharr** | Improved 3×3 derivative |
| Clean general-purpose edge map | **Canny** | Multi-stage + hysteresis |
| Artistic 3-D-like effect | **Emboss kernel** | Emphasizes local differences |
| Smaller output | **Valid convolution** | No padding |
| Same output size | **Same convolution** | Padding |
| Skip pixels while moving kernel | **Stride** | Kernel moves by >1 pixel |

**Memory:** HOG = shape/edges, LBP = texture, Color Histogram = color, HED = edge direction, HIG = intensity gradients.

# 1. HOG — Histogram of Oriented Gradients

HOG represents an image using the **distribution of local gradient directions**.

### Manual workflow
1. Convert image to grayscale.
2. Calculate gradient magnitude/orientation.
3. Divide image into small cells.
4. Build orientation histograms.
5. Group cells into blocks and normalize.
6. Concatenate them into one feature vector.

**Best for:** object detection, pedestrian detection, shape/edge structure.

In [ ]:
# CELL 3 — HOG (MANUAL CODE, WITH DETAILED COMMENTS)

from skimage.feature import hog
from skimage import exposure

image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

# pixels_per_cell=(8,8): each cell is 8 x 8 pixels.
# cells_per_block=(2,2): each normalization block contains 2 x 2 cells.
# visualize=True: return an additional image showing HOG structure.
# feature_vector=True is the default: return one flattened feature vector.
features, hog_image = hog(
    image,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True
)

# Rescale the HOG visualization so it is easier to see.
hog_image_rescaled = exposure.rescale_intensity(
    hog_image,
    in_range=(0, 10)
)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap=plt.cm.gray)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(hog_image_rescaled, cmap=plt.cm.gray)
plt.title("HOG Features")
plt.axis("off")

plt.show()

print("HOG Feature Vector:")
print(features)
print("Feature vector length:", len(features))

### HOG function reference — `skimage.feature.hog()`

Common signature:

`hog(image, orientations=9, pixels_per_cell=(8,8), cells_per_block=(3,3), block_norm='L2-Hys', visualize=False, transform_sqrt=False, feature_vector=True, ...)`

- `image`: input image.
- `orientations`: number of direction bins; default **9**, positive integer.
- `pixels_per_cell`: `(rows, cols)`; default **(8,8)**.
- `cells_per_block`: `(rows, cols)`; default **(3,3)**.
- `block_norm`: valid common values **`'L1'`, `'L1-sqrt'`, `'L2'`, `'L2-Hys'`**; default **`'L2-Hys'`**.
- `visualize`: `False`/`True`; default **False**.
- `transform_sqrt`: `False`/`True`; default **False**.
- `feature_vector`: `False`/`True`; default **True**.

**Parameter effect:** smaller cells preserve more local detail but produce more features. More orientation bins give finer direction information but increase feature size.

# 2. LBP — Local Binary Pattern

LBP is a **texture feature**.

For each center pixel:
- Compare every neighboring pixel with the center.
- Neighbor >= center → **1**
- Neighbor < center → **0**
- The binary pattern becomes the LBP code.
- Count codes in a histogram.
- The histogram becomes the feature vector.

**Best for:** local/repeated textures such as fabric, wood, and surface patterns.

In [ ]:
# CELL 4 — LBP (MANUAL CODE, WITH DETAILED COMMENTS)

from skimage import feature

image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

# Radius = size of the circular neighborhood.
radius = 1

# Manual uses 8 neighboring samples for radius 1.
n_points = 8 * radius

# method='uniform' groups uniform patterns into a compact representation.
lbp_image = feature.local_binary_pattern(
    image,
    n_points,
    radius,
    method='uniform'
)

# Build histogram of LBP pattern occurrences.
hist, _ = np.histogram(
    lbp_image.ravel(),
    bins=np.arange(0, n_points + 3),
    range=(0, n_points + 2)
)

# Convert raw counts into normalized values.
hist = hist.astype(float)
hist /= (hist.sum() + 1e-6)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(lbp_image, cmap='gray')
plt.title("LBP Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(range(0, n_points + 2), hist)
plt.title("LBP Histogram")
plt.xlabel("LBP Patterns")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

print("LBP Feature Vector:")
print(hist)

### LBP function reference — `local_binary_pattern()`

`local_binary_pattern(image, P, R, method='default')`

- `image`: grayscale image.
- `P`: number of sampling points; positive integer.
- `R`: radius; positive number.
- `method`: important valid values:
  - `'default'` — original LBP.
  - `'ror'` — rotation-invariant.
  - `'uniform'` — uniform patterns.
  - `'nri_uniform'` — non-rotation-invariant uniform patterns.
  - `'var'` — local variance.
- Default `method` = **`'default'`**.

**Rule:** larger `P` samples more neighbors; larger `R` examines a larger neighborhood.

### `np.histogram()`
`np.histogram(a, bins=10, range=None, density=None, weights=None)`

- `a`: input data.
- `bins`: default **10**; integer or explicit bin edges.
- `range`: `(min,max)`; default **None**.
- `density`: default **False**; `True` returns a density.
- `weights`: default **None**.

# 3. Histogram of Color

A color histogram describes the distribution of color intensities.

Common spaces:
- **RGB/BGR**: separate histogram for each channel.
- **HSV**: often useful when color information is more important than raw RGB values.

**Best for:** color analysis and color-based comparison/classification.

> The manual explains this technique but does not include a code block, so the following is an added practical implementation.

In [ ]:
# CELL 5 — COLOR HISTOGRAM (PRACTICAL EXAMPLE)

color_image = cv2.imread(IMAGE_PATH)

if color_image is None:
    raise FileNotFoundError(IMAGE_PATH)

# OpenCV stores color as BGR. Convert to RGB for matplotlib.
rgb_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 4))

for channel, name in enumerate(["R", "G", "B"]):
    hist = cv2.calcHist(
        [rgb_image],
        [channel],
        None,
        [256],
        [0, 256]
    )
    plt.plot(hist, label=name)

plt.xlim([0, 256])
plt.xlabel("Intensity")
plt.ylabel("Frequency")
plt.title("RGB Color Histogram")
plt.legend()
plt.show()

# 4. HED — Histogram of Edge Directions

HED captures the **distribution of edge orientations**.

### Workflow
1. Grayscale.
2. Optional Gaussian smoothing.
3. Edge detection.
4. Calculate gradient orientation.
5. Divide angles into bins.
6. Count pixels in each bin.
7. Histogram becomes the feature vector.

**Best for:** dominant edge directions, texture analysis, object recognition and segmentation.

In [ ]:
# CELL 6 — HED (MANUAL CODE)

image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

# Optional smoothing to reduce noise.
# (5,5) = Gaussian kernel size; sigma=0 lets OpenCV choose it.
image_smoothed = cv2.GaussianBlur(image, (5, 5), 0)

# Detect edges.
edges = cv2.Canny(
    image_smoothed,
    threshold1=30,
    threshold2=70
)

# Horizontal and vertical gradients.
gradient_x = cv2.Sobel(
    image_smoothed, cv2.CV_64F, 1, 0, ksize=3
)
gradient_y = cv2.Sobel(
    image_smoothed, cv2.CV_64F, 0, 1, ksize=3
)

# Strength of gradient.
gradient_magnitude = np.sqrt(
    gradient_x**2 + gradient_y**2
)

# Direction of gradient in degrees.
gradient_orientation = (
    np.arctan2(gradient_y, gradient_x) * 180 / np.pi
)

# 8 bins covering 0..360 degrees.
hist, bin_edges = np.histogram(
    gradient_orientation,
    bins=8,
    range=(0, 360)
)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(edges, cmap='gray')
plt.title('Edge Map')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.bar(
    bin_edges[:-1],
    hist,
    width=45,
    align='center'
)
plt.title('Histogram of Edge Directions')
plt.xlabel('Edge Direction (degrees)')
plt.ylabel('Frequency')
plt.xlim(0, 360)

plt.show()

print("HED Feature Vector:")
print(hist)

**Implementation note:** `np.arctan2()` returns approximately -180° to +180°. If you want a clean 0..360 histogram, wrap the angle with:

`orientation = np.mod(np.degrees(np.arctan2(dy, dx)), 360)`

The manual's displayed code is preserved above.

# 5. HIG — Histogram of Intensity Gradients

HIG captures the distribution of **intensity gradients**.

### Formulas

`Magnitude = sqrt(dx² + dy²)`

`Orientation = atan2(dy, dx)`

The manual example uses Sobel, 9 bins, range 0..180, and optional normalization.

**Best for:** object recognition, texture analysis and image classification where gradient information is useful.

In [ ]:
# CELL 7 — HIG (MANUAL CODE)

image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

# Horizontal gradient.
dx = cv2.Sobel(
    image, cv2.CV_64F, 1, 0, ksize=3
)

# Vertical gradient.
dy = cv2.Sobel(
    image, cv2.CV_64F, 0, 1, ksize=3
)

# Gradient magnitude = strength of intensity change.
magnitude = np.sqrt(dx**2 + dy**2)

# Gradient orientation in degrees.
orientation = np.arctan2(dy, dx) * (180 / np.pi)

# Manual: 9 bins from 0 to 180 degrees.
histogram, bins = np.histogram(
    orientation,
    bins=9,
    range=(0, 180)
)

# Normalize the histogram.
histogram = histogram / (histogram.sum() + 1e-12)

plt.bar(
    bins[:-1],
    histogram,
    width=(bins[1] - bins[0])
)
plt.title("Histogram of Intensity Gradients (HIG)")
plt.xlabel("Gradient Orientation (degrees)")
plt.ylabel("Frequency")
plt.show()

print("HIG Feature Vector:")
print(histogram)

# 6. Texture Energy and Texture Contrast

### Texture Energy
The manual defines energy as:

`E = Σ(pixel values)²`

Higher values indicate more complex/less uniform texture in the manual's description.

### Texture Contrast
The manual describes:

`C = sqrt(Var(pixel values))`

which is the local standard deviation.

**Best for:** texture smoothness/complexity and local intensity variation.

In [ ]:
# CELL 8 — TEXTURE ENERGY + CONTRAST

image = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

neighborhood_size = 3

# Square each pixel.
squared_image = image ** 2

# Sum squared values inside each 3x3 neighborhood.
energy = cv2.filter2D(
    squared_image,
    -1,
    np.ones((neighborhood_size, neighborhood_size))
)

# Practical true local standard deviation:
# mean(x^2) - mean(x)^2 = variance
image_f = image.astype(np.float32)

mean_kernel = np.ones(
    (neighborhood_size, neighborhood_size),
    dtype=np.float32
) / (neighborhood_size ** 2)

local_mean = cv2.filter2D(image_f, -1, mean_kernel)
local_mean_square = cv2.filter2D(
    image_f ** 2, -1, mean_kernel
)

local_variance = np.maximum(
    local_mean_square - local_mean ** 2,
    0
)

contrast = np.sqrt(local_variance)

# Histograms.
energy_histogram, energy_bins = np.histogram(
    energy,
    bins=256,
    range=(0, energy.max())
)

contrast_histogram, contrast_bins = np.histogram(
    contrast,
    bins=256,
    range=(0, contrast.max())
)

# Normalize.
energy_histogram = energy_histogram / (
    energy_histogram.sum() + 1e-12
)
contrast_histogram = contrast_histogram / (
    contrast_histogram.sum() + 1e-12
)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(energy_bins[:-1], energy_histogram)
plt.title("Texture Energy Histogram")
plt.xlabel("Energy")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.plot(contrast_bins[:-1], contrast_histogram)
plt.title("Texture Contrast Histogram")
plt.xlabel("Contrast")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

print("Energy feature vector:")
print(energy_histogram)

print("\nContrast feature vector:")
print(contrast_histogram)

# 7. Filtering & Convolution

### Filtering
A filter/kernel is a small matrix applied to image neighborhoods to modify pixel values.

### Convolution
The kernel slides over the image, multiplies corresponding values and sums them.

### Manual examples
- Box blur → simple averaging.
- Gaussian blur → smoother/noise reduction.
- Sobel/Scharr → edge detection.
- Emboss → stylized 3-D-like effect.

In [ ]:
# CELL 9 — BOX BLUR, GAUSSIAN BLUR, SOBEL, SCHARR, EMBOSS

image_color = cv2.imread(IMAGE_PATH)

if image_color is None:
    raise FileNotFoundError(IMAGE_PATH)

# ---------------- BOX BLUR ----------------
# Every value is 1/9, so this is a 3x3 average.
box_kernel = np.ones((3, 3), dtype=np.float32) / 9

blurred_image = cv2.filter2D(
    image_color, -1, box_kernel
)

# ---------------- GAUSSIAN BLUR ----------------
kernel_size = (5, 5)
sigma = 1.0

# getGaussianKernel returns a 1-D normalized Gaussian.
g1 = cv2.getGaussianKernel(kernel_size[0], sigma)

# Make a 2-D Gaussian kernel.
g2 = g1 @ g1.T

gaussian_blur = cv2.filter2D(
    image_color, -1, g2
)

# ---------------- SOBEL ----------------
sobel_x = cv2.Sobel(
    image_color, cv2.CV_64F, 1, 0, ksize=3
)
sobel_y = cv2.Sobel(
    image_color, cv2.CV_64F, 0, 1, ksize=3
)

# ---------------- SCHARR ----------------
scharr_x = cv2.Scharr(
    image_color, cv2.CV_64F, 1, 0
)
scharr_y = cv2.Scharr(
    image_color, cv2.CV_64F, 0, 1
)

# ---------------- EMBOSS ----------------
emboss_kernel = np.array([
    [-2, -1, 0],
    [-1,  1, 1],
    [ 0,  1, 2]
], dtype=np.float32)

embossed_image = cv2.filter2D(
    image_color, -1, emboss_kernel
)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(cv2.cvtColor(blurred_image, cv2.COLOR_BGR2RGB))
plt.title("Box Blur")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(cv2.cvtColor(gaussian_blur, cv2.COLOR_BGR2RGB))
plt.title("Gaussian Blur")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(cv2.cvtColor(embossed_image, cv2.COLOR_BGR2RGB))
plt.title("Emboss")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.imshow(np.abs(sobel_x).astype(np.uint8), cmap="gray")
plt.title("Sobel X")
plt.axis("off")

plt.tight_layout()
plt.show()

# 8. Convolution Types

### Valid / No Padding
Only positions where the kernel fully fits are processed.

For input `H × W`, kernel `K × K`, stride 1:

`H_out = H-K+1`

`W_out = W-K+1`

### Same / Zero Padding
Padding keeps the output spatial size the same for suitable odd kernels and stride 1.

### Valid + Stride
Kernel moves by more than one pixel.

For kernel `(K_h,K_w)` and stride `(S_h,S_w)`:

`H_out = floor((H-K_h)/S_h) + 1`

`W_out = floor((W-K_w)/S_w) + 1`

**Important:** the manual's shown `cv2.filter2D(..., strides=(2,2), ...)` is not a valid OpenCV `filter2D` argument. `filter2D()` has no `strides` parameter. The next cell implements the intended valid/stride operation with NumPy.

In [ ]:
# CELL 10 — STANDARD / VALID / VALID+STRIDE

image_gray = cv2.imread(
    IMAGE_PATH,
    cv2.IMREAD_GRAYSCALE
)

kernel = np.array([
    [1, 0, -1],
    [2, 0, -2],
    [1, 0, -1]
], dtype=np.float32)

# Standard OpenCV filtering.
standard_result = cv2.filter2D(
    image_gray,
    -1,
    kernel
)

def valid_convolution(image, kernel, stride=(1, 1)):
    # Convert to float to avoid integer overflow/truncation.
    image = image.astype(np.float32)
    kernel = kernel.astype(np.float32)

    kh, kw = kernel.shape
    sh, sw = stride

    out_h = (image.shape[0] - kh) // sh + 1
    out_w = (image.shape[1] - kw) // sw + 1

    output = np.zeros((out_h, out_w), dtype=np.float32)

    for i in range(out_h):
        for j in range(out_w):
            y = i * sh
            x = j * sw

            region = image[
                y:y+kh,
                x:x+kw
            ]

            output[i, j] = np.sum(region * kernel)

    return output

# Valid convolution: no padding, stride 1.
valid_result = valid_convolution(
    image_gray, kernel, stride=(1, 1)
)

# Valid convolution with stride 2.
valid_stride2_result = valid_convolution(
    image_gray, kernel, stride=(2, 2)
)

print("Original:", image_gray.shape)
print("Standard:", standard_result.shape)
print("Valid:", valid_result.shape)
print("Valid + stride 2:", valid_stride2_result.shape)

# 9. Edge Detection — Common Techniques

The manual discusses:
- Gradient-based detection
- Canny
- Laplacian of Gaussian (LoG)
- Sobel and Scharr
- Prewitt
- Marr-Hildreth
- Sobel-Feldman
- CNN-based edge detection

### When to choose
- **Sobel:** simple x/y gradient.
- **Scharr:** strong/accurate 3×3 derivative.
- **Canny:** robust clean edge map.
- **LoG/Marr-Hildreth:** second-derivative zero-crossing approach.
- **Prewitt:** simple 3×3 gradient operator.
- **CNN:** learned/complex domain-specific edge patterns.

In [ ]:
# CELL 11 — SOBEL EDGE DETECTION (MANUAL)

image = cv2.imread(
    IMAGE_PATH,
    cv2.IMREAD_GRAYSCALE
)

# x derivative: intensity change along x.
gradient_x = cv2.Sobel(
    image, cv2.CV_64F, 1, 0, ksize=3
)

# y derivative: intensity change along y.
gradient_y = cv2.Sobel(
    image, cv2.CV_64F, 0, 1, ksize=3
)

# Edge strength.
gradient_magnitude = np.sqrt(
    gradient_x**2 + gradient_y**2
)

# Edge/gradient orientation.
gradient_direction = np.arctan2(
    gradient_y,
    gradient_x
)

# Simple thresholding of gradient magnitude.
threshold = 100
edges = (
    gradient_magnitude > threshold
).astype(np.uint8) * 255

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(edges, cmap="gray")
plt.title("Sobel Thresholded Edges")
plt.axis("off")

plt.show()

### `cv2.Sobel()` parameters

`cv2.Sobel(src, ddepth, dx, dy, ksize=3, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)`

- `src`: input image.
- `ddepth`: output depth. Common: `CV_64F`, `CV_32F`, `CV_16S`; `-1` = same source depth.
- `dx`, `dy`: derivative orders; common 0 or 1.
- `ksize`: odd kernel size; common **1, 3, 5, 7**. `-1` selects Scharr.
- `scale`: default **1**.
- `delta`: default **0**.
- `borderType`: default **`BORDER_DEFAULT`** (`BORDER_REFLECT_101`).

**Why `CV_64F`?** Gradients can be negative, so keeping a signed floating-point output is safer.

In [ ]:
# CELL 12 — SCHARR EDGE DETECTION

scharr_x = cv2.Scharr(
    image, cv2.CV_64F, 1, 0
)

scharr_y = cv2.Scharr(
    image, cv2.CV_64F, 0, 1
)

scharr_magnitude = np.sqrt(
    scharr_x**2 + scharr_y**2
)

# Normalize only for display.
scharr_display = cv2.normalize(
    scharr_magnitude,
    None,
    0, 255,
    cv2.NORM_MINMAX
).astype(np.uint8)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(scharr_display, cmap="gray")
plt.title("Scharr Gradient Magnitude")
plt.axis("off")

plt.show()

# 10. Canny Edge Detector

Canny is a **multi-stage edge detector**.

### Stages
1. Gaussian smoothing → reduce noise.
2. Gradient calculation → find intensity changes.
3. Non-maximum suppression → thin edges.
4. Hysteresis → connect useful weak edges to strong edges.

### Thresholds
- `threshold1` = low threshold.
- `threshold2` = high threshold.
- Above high → strong edge.
- Between low/high → weak edge; retained when connected to strong edge.

**Best general-purpose choice when you want a clean edge map with noise suppression.**

In [ ]:
# CELL 13 — CANNY (MANUAL)

image = cv2.imread(
    IMAGE_PATH,
    cv2.IMREAD_GRAYSCALE
)

# Manual explains Gaussian smoothing before Canny.
image_smoothed = cv2.GaussianBlur(
    image,
    (5, 5),
    0
)

# Low threshold = 100, high threshold = 200.
edges = cv2.Canny(
    image_smoothed,
    threshold1=100,
    threshold2=200
)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(edges, cmap="gray")
plt.title("Canny Edges")
plt.axis("off")

plt.show()

### `cv2.Canny()` parameters

Common signature:

`cv2.Canny(image, threshold1, threshold2, apertureSize=3, L2gradient=False)`

- `image`: input image.
- `threshold1`: low hysteresis threshold.
- `threshold2`: high hysteresis threshold.
- `apertureSize`: Sobel aperture; common **3, 5, 7**, default **3**.
- `L2gradient`: `False`/`True`, default **False**.
  - False → approximate gradient magnitude.
  - True → Euclidean magnitude.

### Threshold effect
- Too low → too many/noisy edges.
- Too high → weak edges disappear.
- Start with something like `low=50, high=100` or `100,200`, then adjust to the image.

# 11. Gaussian Blur — Parameters

`cv2.GaussianBlur(src, ksize, sigmaX, sigmaY=0, borderType=cv2.BORDER_DEFAULT)`

- `src`: input image.
- `ksize`: `(width,height)`, normally positive odd values such as `(3,3)`, `(5,5)`, `(7,7)`. `(0,0)` can be used when sigma is explicitly supplied.
- `sigmaX`: X standard deviation; `0` → calculated from kernel size.
- `sigmaY`: Y standard deviation; `0` → same as sigmaX.
- `borderType`: default `BORDER_DEFAULT`.

**Scenario:** more noise → stronger smoothing, but too much smoothing can erase small edges.

# 12. `cv2.filter2D()` — Parameters

`cv2.filter2D(src, ddepth, kernel, dst=None, anchor=(-1,-1), delta=0, borderType=cv2.BORDER_DEFAULT)`

- `src`: input image.
- `ddepth`: output depth; `-1` = same as input.
- `kernel`: filter/convolution kernel.
- `dst`: optional destination; default `None`.
- `anchor`: kernel anchor; `(-1,-1)` = center.
- `delta`: value added after filtering; default `0`.
- `borderType`: default `BORDER_DEFAULT`.

Common border types:
`BORDER_CONSTANT`, `BORDER_REPLICATE`, `BORDER_REFLECT`, `BORDER_REFLECT_101` / `BORDER_DEFAULT`, `BORDER_WRAP`, `BORDER_TRANSPARENT`, `BORDER_ISOLATED`.

**Kernel decides the operation:** averaging → blur; derivative → edges; emboss kernel → stylized effect.

# 13. `cv2.imread()` + Display Functions

### `cv2.imread(filename, flags=cv2.IMREAD_COLOR)`
Important flags:
- `IMREAD_COLOR` = 1 → color/BGR, default.
- `IMREAD_GRAYSCALE` = 0 → grayscale.
- `IMREAD_UNCHANGED` = -1 → load as-is.

Other OpenCV read modes include reduced-size variants such as:
`IMREAD_REDUCED_COLOR_2`, `_4`, `_8` and
`IMREAD_REDUCED_GRAYSCALE_2`, `_4`, `_8`.

### `cv2.imshow(windowName, image)`
Shows an image in an OpenCV window.

### `cv2.waitKey(delay=0)`
- `0` → wait indefinitely.
- positive integer → wait that many milliseconds.
- returns key code, or `-1` if no key was received.

### `cv2.destroyAllWindows()`
Closes all OpenCV windows.

**Jupyter tip:** use `plt.imshow()` when possible because it works more reliably inside notebooks.

# 14. Scharr + Gaussian Kernel

### `cv2.Scharr(src, ddepth, dx, dy, scale=1, delta=0, borderType=cv2.BORDER_DEFAULT)`
- `src`: input.
- `ddepth`: output depth.
- `dx`, `dy`: derivative orders.
- `scale`: default `1`.
- `delta`: default `0`.
- `borderType`: default `BORDER_DEFAULT`.

Scharr is useful for a **3×3 gradient derivative** with improved rotational behavior compared with a basic 3×3 Sobel.

### `cv2.getGaussianKernel(ksize, sigma, ktype=cv2.CV_64F)`
- `ksize`: positive odd size.
- `sigma`: standard deviation; `0` lets OpenCV derive it.
- `ktype`: `CV_32F` or `CV_64F`; default `CV_64F`.

It returns a normalized 1-D Gaussian kernel. A 2-D kernel can be formed with:

`kernel_2d = kernel_1d @ kernel_1d.T`

# 15. Manual Task — Texture Analysis for Material Classification

### Scenario
Classify materials such as **wood, metal and fabric**.

### Recommended features

**LBP**
- Strong for local texture patterns.
- Good for repeated/surface textures.
- Limitation: can be affected by noise, scale and illumination.

**Texture Energy + Contrast**
- Useful for smoothness, activity and local variation.
- Can help separate smooth metal from rough fabric/wood.
- Depends on neighborhood size and lighting.

**HOG**
- Useful as a complementary feature when directional edges/structure matter.
- Limitation: captures gradient/shape information more than pure texture.

### Strong simple baseline
Use:

**LBP histogram + local texture contrast/energy**

Optionally add HOG, concatenate the features, then train a classifier such as SVM, k-NN or Random Forest.

In [ ]:
# CELL 14 — SUGGESTED FEATURE EXTRACTION FOR MATERIAL CLASSIFICATION

def extract_material_features(gray):
    # ---------- LBP ----------
    radius = 1
    n_points = 8 * radius

    lbp = local_binary_pattern(
        gray,
        n_points,
        radius,
        method="uniform"
    )

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2)
    )

    lbp_hist = lbp_hist.astype(np.float32)
    lbp_hist /= (lbp_hist.sum() + 1e-12)

    # ---------- HOG ----------
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        feature_vector=True
    ).astype(np.float32)

    # ---------- Local texture contrast ----------
    gray_f = gray.astype(np.float32)

    kernel = np.ones((3, 3), np.float32) / 9.0

    mean = cv2.filter2D(gray_f, -1, kernel)
    mean_square = cv2.filter2D(
        gray_f ** 2, -1, kernel
    )

    variance = np.maximum(
        mean_square - mean ** 2,
        0
    )

    local_std = np.sqrt(variance)

    # Small global summaries of local contrast.
    texture_summary = np.array([
        np.mean(local_std),
        np.std(local_std)
    ], dtype=np.float32)

    # Final combined feature vector.
    return np.concatenate([
        lbp_hist,
        hog_features,
        texture_summary
    ])


features = extract_material_features(image)

print("Combined feature vector shape:", features.shape)
print("First 20 values:", features[:20])

# 16. FINAL PRACTICAL / EXAM QUICK GUIDE

- **Shape / pedestrian / object structure** → HOG
- **Texture** → LBP
- **Color distribution** → Color Histogram
- **Dominant edge directions** → HED
- **Gradient distribution** → HIG
- **Texture activity** → Energy
- **Local intensity variation** → Contrast / standard deviation
- **Noise reduction** → Gaussian Blur
- **Simple smoothing** → Box Blur
- **Horizontal/vertical gradients** → Sobel
- **Strong 3×3 derivative** → Scharr
- **Clean robust edges** → Canny
- **Artistic 3-D effect** → Emboss
- **No padding** → Valid convolution
- **Same output size** → Same convolution + padding
- **Skip pixels** → Stride

### Key formulas
`G = sqrt(dx² + dy²)`

`θ = atan2(dy, dx)`

`LBP: neighbor >= center → 1, otherwise 0`

`Energy = Σ(pixel values)²`

`Contrast = sqrt(Variance) = standard deviation`

`Valid: H_out = H-K+1, W_out = W-K+1`

`Valid + stride: H_out = floor((H-K)/S)+1`

### Important practical warning
Values such as `(5,5)`, `30,70`, and `100,200` in the manual are **example parameter choices**, not universal fixed values. In scenario questions, adjust them according to noise level, edge strength, image scale and desired output.


# 17. FUNCTION REFERENCE — FUNCTIONS USED IN THIS NOTEBOOK

This section is intentionally compact. For parameters that are numeric/continuous, there is no finite list of "all possible values"; the notebook gives the valid type/range, default, and important choices. For enum-like parameters, the important accepted values are listed.

### OpenCV
**`cv2.imread(filename, flags=cv2.IMREAD_COLOR)`**
- Purpose: read image.
- `flags`: `IMREAD_COLOR=1` default, `IMREAD_GRAYSCALE=0`, `IMREAD_UNCHANGED=-1`, plus reduced-size variants.

**`cv2.cvtColor(src, code)`**
- Purpose: convert color spaces.
- Common `code`: `COLOR_BGR2RGB`, `COLOR_RGB2BGR`, `COLOR_BGR2GRAY`, `COLOR_GRAY2BGR`, `COLOR_BGR2HSV`, etc.

**`cv2.filter2D(src, ddepth, kernel, dst=None, anchor=(-1,-1), delta=0, borderType=BORDER_DEFAULT)`**
- Purpose: apply arbitrary 2-D filter/kernel.
- Important defaults: `dst=None`, `anchor=(-1,-1)`, `delta=0`, `borderType=BORDER_DEFAULT`.

**`cv2.GaussianBlur(src, ksize, sigmaX, sigmaY=0, borderType=BORDER_DEFAULT)`**
- Purpose: Gaussian smoothing.
- `ksize`: positive odd dimensions or `(0,0)`.
- `sigmaX`, `sigmaY`: non-negative; 0 lets OpenCV derive/use the other value.

**`cv2.getGaussianKernel(ksize, sigma, ktype=CV_64F)`**
- Purpose: 1-D Gaussian kernel.
- `ksize`: positive odd integer.
- `sigma`: non-negative; 0 = derive from size.
- `ktype`: `CV_32F` or `CV_64F`.

**`cv2.Sobel(src, ddepth, dx, dy, ksize=3, scale=1, delta=0, borderType=BORDER_DEFAULT)`**
- Purpose: derivative/gradient.
- `dx`, `dy`: derivative orders.
- `ksize`: common 1,3,5,7; `-1` = Scharr.
- `scale=1`, `delta=0`, default border.

**`cv2.Scharr(src, ddepth, dx, dy, scale=1, delta=0, borderType=BORDER_DEFAULT)`**
- Purpose: 3x3 Scharr derivative.
- Main parameters same idea as Sobel.

**`cv2.Canny(image, threshold1, threshold2, apertureSize=3, L2gradient=False)`**
- Purpose: multi-stage edge detection.
- `apertureSize`: common 3,5,7.
- `L2gradient`: False/True.

**`cv2.normalize(src, dst, alpha, beta, norm_type, dtype=-1, mask=None)`**
- Purpose: rescale values to a chosen range.
- Common `norm_type`: `NORM_MINMAX`, `NORM_L1`, `NORM_L2`, `NORM_INF`.
- `alpha`, `beta`: target range for `NORM_MINMAX`.

**`cv2.calcHist(images, channels, mask, histSize, ranges, accumulate=False)`**
- Purpose: calculate image histogram.
- `channels`: channel indices.
- `histSize`: number of bins.
- `ranges`: value range, e.g. `[0,256]`.
- `accumulate`: False by default.

**`cv2.imshow(windowName, mat)`** → show OpenCV window.

**`cv2.waitKey(delay=0)`** → wait for key; 0 means indefinitely.

**`cv2.destroyAllWindows()`** → close all OpenCV windows.

### scikit-image
**`hog(...)`**
- Purpose: HOG feature extraction.
- Key defaults: `orientations=9`, `pixels_per_cell=(8,8)`, `cells_per_block=(3,3)`, `block_norm='L2-Hys'`, `visualize=False`, `transform_sqrt=False`, `feature_vector=True`.

**`local_binary_pattern(image, P, R, method='default')`**
- Purpose: LBP texture descriptor.
- Methods: `default`, `ror`, `uniform`, `nri_uniform`, `var`.
- Default method: `default`.

**`exposure.rescale_intensity(image, in_range='image', out_range='dtype')`**
- Purpose: remap intensity range for display/processing.
- `in_range`: `'image'`, `'dtype'`, or a 2-value range.
- `out_range`: `'dtype'`, or a 2-value range.

### NumPy
**`np.histogram(a, bins=10, range=None, density=None, weights=None)`**
→ histogram + bin edges.

**`np.sqrt(x)`** → element-wise square root.

**`np.arctan2(y, x)`** → angle in radians, preserving quadrant.

**`np.degrees(x)`** → radians to degrees.

**`np.ones(shape, dtype=None)`** → array filled with 1s.

**`np.array(object, dtype=None)`** → create NumPy array.

**`np.concatenate(arrays, axis=0)`** → join arrays along an axis.

**`np.maximum(x1, x2)`** → element-wise maximum.

**`np.mean(a, axis=None)`** → arithmetic mean.

**`np.std(a, axis=None)`** → standard deviation.

**`array.ravel()`** → flatten to 1-D view when possible.

### Matplotlib
**`plt.figure(figsize=(w,h))`** → create figure.

**`plt.subplot(nrows,ncols,index)`** → select subplot.

**`plt.imshow(X, cmap=None)`** → display image; `cmap='gray'` for grayscale.

**`plt.plot(x,y)`** → line plot.

**`plt.bar(x,height,width=None,align='center')`** → bar chart.

**`plt.xlabel(text)` / `plt.ylabel(text)`** → axis labels.

**`plt.title(text)`** → title.

**`plt.legend()`** → legend.

**`plt.xlim(left,right)`** → x-axis limits.

**`plt.tight_layout()`** → reduce overlapping layout.

**`plt.show()`** → render the figure.
